# ✅ Write-Audit-Publish (WAP) — Safe Data Ingestion with Iceberg Branches

The WAP pattern lets you **stage data on a branch**, run quality checks, and only **publish to production** if everything passes — all without copying data or locking the table.

```
                        Write-Audit-Publish Flow
───────────────────────────────────────────────────────────────────

  main branch (production)         staging branch (isolated)
  ┌──────────────────────┐         ┌──────────────────────────┐
  │ Snapshot A           │         │ Snapshot A (forked)      │
  │  (users read this)   │         │  + new staged data       │
  └──────────┬───────────┘         └──────────┬───────────────┘
             │                                │
             │          ┌──────────┐          │
             │          │  AUDIT   │◄─────────┘
             │          │ (quality │
             │          │  checks) │
             │          └────┬─────┘
             │               │
             │          Pass? │ Fail?
             │          ┌────┴─────┐
             │     ┌────▼───┐  ┌───▼────┐
             │     │PUBLISH │  │DISCARD │
             │     │fast-fwd│  │drop    │
             │     └────┬───┘  │branch  │
             │          │      └────────┘
  ┌──────────▼──────────▼┐
  │ Snapshot B            │
  │  (new production)     │
  └───────────────────────┘
```

| Step | What Happens | Iceberg Mechanism |
|------|-------------|-------------------|
| **Write** | Ingest data to isolated branch | `INSERT INTO ... FOR VERSION AS OF 'branch'` |
| **Audit** | Run quality checks on branch | `SELECT ... FOR VERSION AS OF 'branch'` |
| **Publish** | Fast-forward main → branch | `CALL system.fast_forward(...)` |

**Key benefits over traditional staging tables:**
- **Zero data duplication** — branches share underlying data files
- **Atomic promotion** — fast-forward is a metadata-only operation
- **Production isolation** — readers on `main` never see staged data
- **Full rollback** — just drop the branch if audits fail

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Trino

In [ ]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="silver",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

---
# Part 1 — Setting Up the Production Table

We'll create a `transactions` table in the silver layer representing a production financial dataset. This is the table that downstream dashboards and reports read from — we can't afford to publish bad data here.

In [ ]:
run_query("DROP TABLE IF EXISTS iceberg.silver.transactions")

run_query("""
CREATE TABLE iceberg.silver.transactions (
    txn_id      BIGINT,
    account_id  VARCHAR,
    amount      DOUBLE,
    currency    VARCHAR,
    txn_date    DATE,
    status      VARCHAR
) WITH (format = 'PARQUET')
""")
print("✅ Table 'transactions' created in silver layer")

### 📥 Seed Production Data

Load a baseline of known-good transactions. This represents the current production state.

In [ ]:
run_query("""
INSERT INTO iceberg.silver.transactions VALUES
    (1, 'ACC-001', 1500.00, 'USD', DATE '2026-03-01', 'completed'),
    (2, 'ACC-002',  250.75, 'USD', DATE '2026-03-01', 'completed'),
    (3, 'ACC-001',  890.00, 'EUR', DATE '2026-03-02', 'completed'),
    (4, 'ACC-003', 3200.00, 'USD', DATE '2026-03-02', 'completed'),
    (5, 'ACC-002',  175.50, 'GBP', DATE '2026-03-03', 'completed')
""")
print("✅ 5 baseline transactions loaded")

In [ ]:
print("📋 Production data (main branch):")
print()
run_query("SELECT * FROM iceberg.silver.transactions ORDER BY txn_id");

---
# Part 2 — The WAP Pattern: Happy Path 🟢

A new batch of transactions arrives from an upstream system. Before publishing to production, we need to ensure data quality. Here's the full Write → Audit → Publish cycle.

## 2.1 — Write: Create a Staging Branch

Create an isolated branch forked from `main`. This is a **metadata-only** operation — no data is copied. The branch starts at the same snapshot as `main`.

In [ ]:
run_query("ALTER TABLE iceberg.silver.transactions CREATE BRANCH audit_batch_01")
print("✅ Branch 'audit_batch_01' created — forked from main")

In [ ]:
print("📋 Table refs — main + new branch share the same snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM iceberg.silver."transactions$refs"
""");

### 📥 Write New Data to the Staging Branch

Insert the incoming batch **only** onto the staging branch. Production readers on `main` won't see any of these rows.

In [ ]:
run_query("""
INSERT INTO iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_01'
VALUES
    (6, 'ACC-001',  420.00, 'USD', DATE '2026-03-04', 'completed'),
    (7, 'ACC-004', 1100.00, 'EUR', DATE '2026-03-04', 'completed'),
    (8, 'ACC-002',  330.25, 'USD', DATE '2026-03-05', 'completed'),
    (9, 'ACC-003',  750.00, 'GBP', DATE '2026-03-05', 'completed')
""")
print("✅ 4 new transactions written to staging branch")

### 🔍 Verify Branch Isolation

The key feature: production (`main`) and staging (`audit_batch_01`) show **different data**. Downstream consumers are completely unaffected.

In [ ]:
print("📋 Production (main) — still 5 rows, unchanged:")
prod_rows = run_query("SELECT count(*) AS row_count FROM iceberg.silver.transactions")
print()
print("📋 Staging branch — 9 rows (5 original + 4 new):")
stage_rows = run_query("""
SELECT count(*) AS row_count
FROM iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_01'
""")
print()
print(f"🔒 Branch isolation confirmed: main={prod_rows[0][0]}, staging={stage_rows[0][0]}")

In [ ]:
print("📋 Refs after write — staging branch has advanced to a new snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM iceberg.silver."transactions$refs"
""");

## 2.2 — Audit: Data Quality Checks on the Branch

Now we run validation queries **against the staging branch only**. These checks ensure the new batch meets our data quality requirements before it reaches production.

| Check | Rule | Threshold |
|-------|------|-----------|
| No NULL amounts | `amount IS NOT NULL` | 0 violations |
| No negative amounts | `amount > 0` | 0 violations |
| Valid currency codes | `currency IN ('USD','EUR','GBP')` | 0 violations |
| No duplicate txn_ids | `COUNT(DISTINCT txn_id) = COUNT(*)` | exact match |

In [ ]:
def audit_branch(branch_name):
    """Run data quality checks against a staging branch. Returns True if all pass."""
    checks = [
        (
            "No NULL amounts",
            f"""
            SELECT count(*) AS violations
            FROM iceberg.silver.transactions FOR VERSION AS OF '{branch_name}'
            WHERE amount IS NULL
            """,
            lambda rows: rows[0][0] == 0,
        ),
        (
            "No negative amounts",
            f"""
            SELECT count(*) AS violations
            FROM iceberg.silver.transactions FOR VERSION AS OF '{branch_name}'
            WHERE amount <= 0
            """,
            lambda rows: rows[0][0] == 0,
        ),
        (
            "Valid currency codes",
            f"""
            SELECT count(*) AS violations
            FROM iceberg.silver.transactions FOR VERSION AS OF '{branch_name}'
            WHERE currency NOT IN ('USD', 'EUR', 'GBP')
            """,
            lambda rows: rows[0][0] == 0,
        ),
        (
            "No duplicate txn_ids",
            f"""
            SELECT
                count(*) AS total,
                count(DISTINCT txn_id) AS distinct_ids
            FROM iceberg.silver.transactions FOR VERSION AS OF '{branch_name}'
            """,
            lambda rows: rows[0][0] == rows[0][1],
        ),
    ]

    all_passed = True
    for name, sql, check_fn in checks:
        result = run_query(sql, display=False)
        passed = check_fn(result)
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {status} — {name}")
        if not passed:
            all_passed = False

    return all_passed


print(f"🔍 Auditing branch 'audit_batch_01'...")
print()
passed = audit_branch("audit_batch_01")
print()
print(f"{'🟢 All checks passed — safe to publish!' if passed else '🔴 Audit failed — do NOT publish!'}")

## 2.3 — Publish: Fast-Forward Main to the Branch

All checks passed! Now we **fast-forward** `main` to point to the staging branch's latest snapshot. This is:
- **Atomic** — readers instantly see all new data or none of it
- **Metadata-only** — no data files are moved or copied
- **Instantaneous** — regardless of data size

In [ ]:
run_query("CALL iceberg.system.fast_forward('silver', 'transactions', 'main', 'audit_batch_01')")
print("✅ Published! main branch fast-forwarded to audit_batch_01")

In [ ]:
print("📋 Production data after publish — all 9 transactions now visible:")
print()
run_query("SELECT * FROM iceberg.silver.transactions ORDER BY txn_id");

In [ ]:
print("📋 Refs after publish — main and branch now point to the same snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM iceberg.silver."transactions$refs"
""");

### 🧹 Clean Up the Staging Branch

After a successful publish, the staging branch is no longer needed. Drop it to keep metadata clean.

In [ ]:
run_query("ALTER TABLE iceberg.silver.transactions DROP BRANCH audit_batch_01")
print("✅ Branch 'audit_batch_01' dropped — publish complete")

---
# Part 3 — The WAP Pattern: Rejected Batch 🔴

Now let's see what happens when bad data arrives. The audit step catches the issues, and we **discard** the branch — production is never affected.

## 3.1 — Write: Stage a Bad Batch

In [ ]:
run_query("ALTER TABLE iceberg.silver.transactions CREATE BRANCH audit_batch_02")
print("✅ Branch 'audit_batch_02' created")

In [ ]:
run_query("""
INSERT INTO iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_02'
VALUES
    (10, 'ACC-001',  500.00, 'USD', DATE '2026-03-06', 'completed'),
    (11, 'ACC-005', -200.00, 'USD', DATE '2026-03-06', 'completed'),
    (12, 'ACC-002',  680.00, 'XYZ', DATE '2026-03-06', 'completed'),
    (10, 'ACC-001',  500.00, 'USD', DATE '2026-03-06', 'completed')
""")
print("✅ 4 transactions written to staging (includes bad data!)")
print("   ⚠️  txn 11 has negative amount")
print("   ⚠️  txn 12 has invalid currency 'XYZ'")
print("   ⚠️  txn 10 is duplicated")

## 3.2 — Audit: Checks Fail

In [ ]:
print(f"🔍 Auditing branch 'audit_batch_02'...")
print()
passed = audit_branch("audit_batch_02")
print()
print(f"{'🟢 All checks passed — safe to publish!' if passed else '🔴 Audit FAILED — batch will be discarded!'}")

### 🔍 Inspect the Violations

Let's see exactly what failed — querying the branch to understand the issues before discarding it.

In [ ]:
print("❌ Negative amounts:")
run_query("""
SELECT txn_id, account_id, amount
FROM iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_02'
WHERE amount <= 0
""")

print()
print("❌ Invalid currencies:")
run_query("""
SELECT txn_id, account_id, currency
FROM iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_02'
WHERE currency NOT IN ('USD', 'EUR', 'GBP')
""")

print()
print("❌ Duplicate txn_ids:")
run_query("""
SELECT txn_id, count(*) AS occurrences
FROM iceberg.silver.transactions FOR VERSION AS OF 'audit_batch_02'
GROUP BY txn_id
HAVING count(*) > 1
""");

## 3.3 — Discard: Drop the Failed Branch

Since the audit failed, we simply **drop the branch**. Production data on `main` is completely unaffected — as if the bad batch never existed.

In [ ]:
run_query("ALTER TABLE iceberg.silver.transactions DROP BRANCH audit_batch_02")
print("🗑️  Branch 'audit_batch_02' dropped — bad batch discarded")

In [ ]:
print("📋 Production data — still exactly 9 rows, untouched:")
print()
run_query("SELECT * FROM iceberg.silver.transactions ORDER BY txn_id");

In [ ]:
print("📋 Only 'main' ref remains — staging branch is gone:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM iceberg.silver."transactions$refs"
""");

---
# Part 4 — Under the Hood: Snapshots & Branches

Let's inspect the Iceberg metadata to understand what happened at the snapshot level throughout the entire WAP lifecycle.

In [ ]:
print("📸 Full snapshot history — shows the complete WAP lifecycle:")
print()
run_query("""
SELECT committed_at, snapshot_id, parent_id, operation
FROM iceberg.silver."transactions$snapshots"
ORDER BY committed_at
""");

In [ ]:
print("📁 Data files — branches share files, no duplication:")
print()
run_query("""
SELECT
    record_count,
    file_size_in_bytes,
    file_format
FROM iceberg.silver."transactions$files"
""");

---
## 📊 Summary

| Concept | Implementation | Key Benefit |
|---------|---------------|-------------|
| **Write** | `CREATE BRANCH` + `INSERT FOR VERSION AS OF` | Isolated staging, zero duplication |
| **Audit** | `SELECT FOR VERSION AS OF` with validation logic | Query staged data without affecting production |
| **Publish** | `CALL system.fast_forward(main, branch)` | Atomic, metadata-only promotion |
| **Reject** | `DROP BRANCH` | Production untouched, instant cleanup |

```
WAP vs Traditional Staging
──────────────────────────────────────────────────────────────────
                        Traditional          WAP + Branching
  Data duplication      Full copy to         Zero — shared files
                        staging table
  Publish               INSERT + DELETE      Metadata pointer swap
  Rollback              Manual cleanup       DROP BRANCH
  Isolation             Separate table       Same table, diff branch
  Atomicity             Multi-step           Single fast-forward
```

**Production considerations:**
- Automate WAP cycles via Airflow/Dagster — create branch → ingest → audit → publish/reject
- Use branch naming conventions like `audit_<date>_<batch_id>` for traceability
- Add branch TTLs to auto-expire forgotten staging branches
- Combine with CDC snapshotting for end-to-end safe ingestion pipelines

---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop the table:
# run_query("DROP TABLE IF EXISTS iceberg.silver.transactions")
# print("🗑️ Table 'transactions' dropped")